In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")

In [ ]:
import re
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples before filtering: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
stopwords = {
    "a", "an", "the", "and", "or", "but", "if", "while", "of", "at", "by", "for", "with", "about", "against",
    "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down",
    "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when",
    "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no",
    "nor", "not", "only", "own", "same", "so", "than", "too", "very", "can", "will", "just", "is", "am", "are",
    "was", "were", "be", "been", "being", "have", "has", "had", "do", "does", "did", "this", "that", "these",
    "those", "as", "it", "its", "he", "she", "they", "them", "his", "her", "their", "you", "your", "we", "our",
    "i", "me", "my"
}

def simple_tokens(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return [t for t in tokens if t not in stopwords and len(t) > 1]

def overlap_stats(s1, s2):
    t1 = set(simple_tokens(s1))
    t2 = set(simple_tokens(s2))
    inter = t1 & t2
    union = t1 | t2
    jaccard = len(inter) / len(union) if union else 0.0
    containment_min = len(inter) / min(len(t1), len(t2)) if min(len(t1), len(t2)) > 0 else 0.0
    return {
        "tokens1": t1,
        "tokens2": t2,
        "intersection": inter,
        "union": union,
        "jaccard": jaccard,
        "containment_min": containment_min,
        "len1": len(t1),
        "len2": len(t2),
        "overlap_count": len(inter)
    }

hard_rows = []
all_jaccards = []
all_containments = []

for idx, row in enumerate(dataset):
    stats = overlap_stats(row["sentence1"], row["sentence2"])
    all_jaccards.append(stats["jaccard"])
    all_containments.append(stats["containment_min"])
    label = row["label"]
    high_overlap_negative = (label == 0 and stats["jaccard"] >= 0.5)
    low_overlap_positive = (label == 1 and stats["jaccard"] <= 0.2)
    if high_overlap_negative or low_overlap_positive:
        enriched = dict(row)
        enriched["orig_idx"] = idx
        enriched["jaccard"] = stats["jaccard"]
        enriched["containment_min"] = stats["containment_min"]
        enriched["overlap_count"] = stats["overlap_count"]
        enriched["len1"] = stats["len1"]
        enriched["len2"] = stats["len2"]
        enriched["overlap_tokens"] = sorted(list(stats["intersection"]))
        enriched["challenge_type"] = "high_overlap_negative" if high_overlap_negative else "low_overlap_positive"
        hard_rows.append(enriched)

print(f"Computed lexical-overlap heuristics for {len(dataset)} validation examples.")
print(f"Hard subset size: {len(hard_rows)}")
print(f"Challenge type counts: {Counter([r['challenge_type'] for r in hard_rows])}")
if hard_rows:
    print("Sample hard example:")
    sample = hard_rows[0]
    print({k: sample[k] for k in ["orig_idx", "label", "challenge_type", "jaccard", "containment_min", "overlap_count"]})

In [ ]:
if len(hard_rows) == 0:
    raise ValueError("Hard subset is empty. Adjust heuristic thresholds if needed.")

sent1_list = [r["sentence1"] for r in hard_rows]
sent2_list = [r["sentence2"] for r in hard_rows]
labels = [r["label"] for r in hard_rows]

batch_size = 32
predictions = []
confidences = []
probabilities = []

for start_idx in range(0, len(hard_rows), batch_size):
    batch_s1 = sent1_list[start_idx:start_idx + batch_size]
    batch_s2 = sent2_list[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch_s1,
        batch_s2,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())
    probabilities.extend(probs.cpu().tolist())

print(f"Completed inference for {len(predictions)} hard-subset examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

subset_jaccards = [r["jaccard"] for r in hard_rows]
subset_containments = [r["containment_min"] for r in hard_rows]
subset_overlap_counts = [r["overlap_count"] for r in hard_rows]

high_overlap_negative_count = sum(r["challenge_type"] == "high_overlap_negative" for r in hard_rows)
low_overlap_positive_count = sum(r["challenge_type"] == "low_overlap_positive" for r in hard_rows)

print("Evaluation metrics on challenge subset:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print()
print("Overlap statistics:")
print(f"Full validation mean jaccard          : {sum(all_jaccards) / len(all_jaccards):.4f}")
print(f"Hard subset mean jaccard              : {sum(subset_jaccards) / len(subset_jaccards):.4f}")
print(f"Hard subset min/max jaccard           : {min(subset_jaccards):.4f} / {max(subset_jaccards):.4f}")
print(f"Hard subset mean containment(min-set) : {sum(subset_containments) / len(subset_containments):.4f}")
print(f"Hard subset mean overlap token count  : {sum(subset_overlap_counts) / len(subset_overlap_counts):.2f}")
print(f"high_overlap_negative count           : {high_overlap_negative_count}")
print(f"low_overlap_positive count            : {low_overlap_positive_count}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

mistakes = []
for i, row in enumerate(hard_rows):
    if predictions[i] != labels[i]:
        mistakes.append({
            "subset_idx": i,
            "orig_idx": row["orig_idx"],
            "true_label": labels[i],
            "pred_label": predictions[i],
            "confidence": confidences[i],
            "prob_not_paraphrase": probabilities[i][0],
            "prob_paraphrase": probabilities[i][1],
            "challenge_type": row["challenge_type"],
            "jaccard": row["jaccard"],
            "containment_min": row["containment_min"],
            "overlap_count": row["overlap_count"],
            "overlap_tokens": row["overlap_tokens"],
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"]
        })

mistakes = sorted(mistakes, key=lambda x: x["confidence"], reverse=True)
print(f"Total mistakes on challenge subset: {len(mistakes)}")

num_examples_to_show = min(10, len(mistakes))
for i in range(num_examples_to_show):
    m = mistakes[i]
    print(f"Mistake {i + 1}")
    print(f"orig_idx: {m['orig_idx']} | challenge_type: {m['challenge_type']}")
    print(f"true label: {m['true_label']} ({label_map[m['true_label']]})")
    print(f"pred label: {m['pred_label']} ({label_map[m['pred_label']]})")
    print(f"confidence: {m['confidence']:.4f}")
    print(f"p(not_paraphrase)={m['prob_not_paraphrase']:.4f} | p(paraphrase)={m['prob_paraphrase']:.4f}")
    print(f"jaccard={m['jaccard']:.4f} | containment_min={m['containment_min']:.4f} | overlap_count={m['overlap_count']}")
    print(f"overlap_tokens: {m['overlap_tokens'][:20]}")
    print(f"sentence1: {m['sentence1']}")
    print(f"sentence2: {m['sentence2']}")
    print("-" * 100)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset=lexical-overlap challenge subset")
print(f"device={device}")
print(f"num_examples_full={len(dataset)}")
print(f"num_examples_subset={len(hard_rows)}")
print(f"high_overlap_negative_count={high_overlap_negative_count}")
print(f"low_overlap_positive_count={low_overlap_positive_count}")
print(f"mean_jaccard_full={sum(all_jaccards) / len(all_jaccards):.4f}")
print(f"mean_jaccard_subset={sum(subset_jaccards) / len(subset_jaccards):.4f}")
print(f"mean_containment_subset={sum(subset_containments) / len(subset_containments):.4f}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"num_mistakes={len(mistakes)}")